In [1]:
import numpy as np
from collections import deque
import heapq, itertools
import random
from typing import Self

In [2]:
def sample_priority():
    return random.choices(PRIORITY, weights=PROBS_PRIORITY)[0]


def sample_geom(mean):
    return int(np.random.geometric(1.0 / mean))

In [ ]:
class Client():
    
    _incrementalID = 0
    
    def __init__(self, priority: int, arrivalTime: int, departureID: int | None):
        self.priority = priority
        self.clientID = Client._incrementalID
        Client._incrementalID += 1
        self.arrivalTime = arrivalTime
        self.remainingTime = -1
        self.departureID = departureID
        

    def sortKey(self) -> tuple[int, int, int, Self]:
        
        return self.priority, self.arrivalTime, self.clientID, self
        

In [ ]:
class Event():
    
    _incrementalID = 0
    
    def __init__(self, time: int, eventType: str, client: Client | None):
        self.time = time
        self.eventType = eventType
        self.eventID = self._incrementalID
        self._incrementalID += 1
        self.client = client
        
        
    def sortKey(self) -> tuple[int, int, Self]:
        return self.time, self.eventID, self
        

In [ ]:
class FutureEventSet():
    
    def __init__(self):
        self.heap = []
        
        
    def insertEvent(self, event: Event):
        heapq.heappush(self.heap, event.sortKey())
        
        
    def getEvent(self):
        event = heapq.heappop(self.heap)
        return event.toEvent()
    
    
    def acceptDeparture(self, clientID: int, departureID: int, event: Event):
        
        if event.client is not None:
            if clientID == event.client.clientID and departureID == event.client.departureID:
                return True
        
        return False

In [ ]:
class UsersQueue:
    def __init__(self, policy: str):
        self.policy = policy
        
        if self.policy == "priority":
            self.queue = []
            
        elif self.policy == "FIFO":
            self.queue = deque()
            
        elif self.policy == "LIFO":
            self.queue = deque()
            
        self.counter = itertools.count()


    def insertUser(self, user: Client) -> None:
        if self.policy == "priority":
            heapq.heappush(self.queue, user.sortKey())  # type: ignore
            
        elif self.policy == "FIFO":
            self.queue.append(user)
            
        elif self.policy == "LIFO":
            self.queue.append(user)


    def getUser(self) -> Client:
        if self.policy == "priority":
            client = heapq.heappop(self.queue)[-1]  # type: ignore
            return client
        
        elif self.policy == "FIFO":
            return self.queue.popleft() # type: ignore
        
        elif self.policy == "LIFO":
            return self.queue.pop()
            
        else:
            return self.queue.pop()
        
        
    def __len__(self):
        return len(self.queue)

In [ ]:
class ServingQueue():
    
    def __init__(self, maxSize: int):
        self.queue = []
        self.maxSize = maxSize
        
    
    def insertClient(self, user: Client):
        if len(self.queue) >= self.maxSize:
            print("C'è qualcosa che non va")

In [ ]:
def arrival(time: int, FES: FutureEventSet, usersQueue: UsersQueue) -> None:
    global users
    
    arrival_time = time + sample_geom(ARRIVAL)
    arrival = Event(arrival_time, "arrival", None)
    FES.insertEvent(arrival)
    
    users += 1 # type: ignore
    
    clientType = sample_priority()
    user = Client(clientType, time, None)
    
    if users <= N_SERVERS:
        serviceTime = sample_geom(SERVICE)
        departureTime = time + serviceTime
        
        departure = Event(departureTime, "departure", user)
        
        user.departureID = departure.eventID
        user.remainingTime = serviceTime
        
        FES.insertEvent(departure)
        
    else:
        usersQueue.insertUser(user)
        
        
        
def departure(time: int, FES: FutureEventSet, usersQueue: UsersQueue, counter) -> None:
    global users
    
    if len(usersQueue) > 0:
        client = usersQueue.getUser()
    users -= 1 # type: ignore
    
    if users >= N_SERVERS:
        service_time = time + sample_geom(SERVICE)
        FES.insertEvent((service_time, "departure"))

In [8]:
ARRIVAL = 5
SERVICE = 8
users = 0
MAX_TIME = 1000
N_SERVERS = 2
PRIORITY = [0]              # 0 = priorità più alta
PROBS_PRIORITY = [1]  

In [9]:
users = 0
FES = FutureEventSet()
usersQueue = UsersQueue(policy="LIFO")
time = 0
FES.insertEvent((0, "arrival"))


while FES.heap:
    time, event_type = FES.getEvent()
    
    if time > MAX_TIME:
        break
    
    if event_type == "arrival":
        arrival(time, FES, usersQueue, None)
    
    elif event_type == "departure":
        departure(time, FES,usersQueue, None)
        
    print(f"{event_type} --- Time: {time} --- len(Usersqueue): {len(usersQueue)} --- users: {users}\n") # type: ignore

AttributeError: 'tuple' object has no attribute 'toTuple'